## TRANSFER LEARNING RAZA GATOS

In [1]:
########################################################################
########## CLASIFICACIÓN DE RAZAS DE GATOS - TRANSFER LEARNING ########
########################################################################

import os
from pathlib import Path

DATA_DIR = Path(os.environ.get("DATA_DIR", "./data"))
MODELS_DIR = Path(os.environ.get("MODELS_DIR", "./models"))
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# ============================================================
# PASO 0. RUTA DEL DATASET
# ============================================================
ruta_gatos = DATA_DIR / "OXFORD" / "GATOS"

# UNA SOLA RUTA PARA EL MEJOR MODELO
ruta_modelo = MODELS_DIR / "modelo_razas_gatos_TransferLearning.keras"

IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32
SEED = 42

# ============================================================
# PASO 1. CARGA DEL DATASET
# ============================================================
dataset_completo = tf.keras.utils.image_dataset_from_directory(
    ruta_gatos,
    seed=SEED,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    shuffle=True
)

class_names = dataset_completo.class_names
num_classes = len(class_names)

print("Razas de gatos:", class_names)
print("Número de clases:", num_classes)

# ============================================================
# PASO 2. SEPARAR TRAIN / VALIDATION / TEST
# ============================================================
total_batches = tf.data.experimental.cardinality(dataset_completo).numpy()

train_size = int(0.70 * total_batches)
val_size = int(0.15 * total_batches)
test_size = total_batches - train_size - val_size

train_dataset = dataset_completo.take(train_size)
resto_dataset = dataset_completo.skip(train_size)

val_dataset = resto_dataset.take(val_size)
test_dataset = resto_dataset.skip(val_size)

print("\nLotes totales:", total_batches)
print("Lotes train:", tf.data.experimental.cardinality(train_dataset).numpy())
print("Lotes validation:", tf.data.experimental.cardinality(val_dataset).numpy())
print("Lotes test:", tf.data.experimental.cardinality(test_dataset).numpy())

# ============================================================
# PASO 3. OPTIMIZAR DATASET
# ============================================================
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.cache().prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.cache().prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.cache().prefetch(buffer_size=AUTOTUNE)

# ============================================================
# PASO 4. DATA AUGMENTATION
# ============================================================
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.10),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomContrast(0.10)
])

# ============================================================
# PASO 5. MODELO BASE PREENTRENADO
# ============================================================
# Aquí se carga MobileNetV2, una red neuronal convolucional ya entrenada previamente con ImageNet.
# ImageNet contiene millones de imágenes de miles de categorías, por lo que el modelo ya aprendió patrones visuales generales.
# include_top=False No carga la capa final original de clasificación. Se elimina la parte que clasificaba clases de ImageNet para reemplazarla por una nueva adaptada a tu problema.
# Es decir:conservas extractor de características, cambias clasificador final.
# weights="imagenet" Carga pesos aprendidos previamente en ImageNet. Esto evita entrenar desde cero.
# Congela todas las capas del modelo base. Significa que sus pesos NO se modifican durante la primera fase de entrenamiento. Solo entrenarás las capas nuevas agregadas arriba.

base_model = MobileNetV2(
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

# ============================================================
# PASO 6. MODELO FINAL
# ============================================================
model = Sequential([
    tf.keras.layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)), # Entrada del modelo.Recibe imágenes RGB con tamaño definido.(224,224,3)
    data_augmentation, # Aplica aumentos aleatorios: volteo, rotación, zoom, contraste. Mejora generalización.
    tf.keras.layers.Lambda(preprocess_input),# Preprocesa pixeles al formato esperado por MobileNetV2.Ajusta valores numéricos según ImageNet.
    base_model, # Pasa la imagen por MobileNetV2. Extrae características visuales profundas.Salida típica aproximada:(7,7,1280)
    GlobalAveragePooling2D(), # Convierte cada mapa de activación en un solo valor promedio. Si entrada = (7,7,1280) salida = (1280) Reduce parámetros y evita overfitting.
    Dropout(0.30),  # Apaga aleatoriamente 30% de neuronas en entrenamiento.
    Dense(128, activation="relu"), # Capa densa de 128 neuronas. Combina características extraídas.
    Dropout(0.30), # Apaga aleatoriamente 30% de neuronas en entrenamiento.
    Dense(num_classes, activation="softmax") # Capa final de clasificación multiclase. Si hay 12 razas: salida = 12 probabilidades Softmax convierte valores en probabilidades cuya suma total = 1.
])

# ============================================================
# PASO 7. COMPILACIÓN
# ============================================================
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# 🔥 VER ARQUITECTURA DEL MODELO
model.summary()

# ============================================================
# PASO 8. CALLBACKS
# ============================================================
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

checkpoint = ModelCheckpoint(
    filepath=ruta_modelo,
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=False,
    mode="min",
    verbose=1
)

# ============================================================
# PASO 9. ENTRENAMIENTO
# ============================================================
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=30,
    callbacks=[early_stopping, reduce_lr, checkpoint],
    verbose=1
)

# ============================================================
# PASO 9.1 CARGAR EL MEJOR MODELO
# ============================================================
print("\nCargando el mejor modelo guardado...")
model = load_model(
    ruta_modelo,
    custom_objects={"preprocess_input": preprocess_input},
    compile=False
)


# 🔥 SOLUCIÓN: recompilar modelo
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


# ============================================================
# PASO 10. EVALUACIÓN FINAL EN TEST
# ============================================================
test_loss, test_acc = model.evaluate(test_dataset, verbose=0)
print("\nTest loss:", test_loss)
print("Test accuracy:", test_acc)

# ============================================================
# PASO 11. MATRIZ DE CONFUSIÓN Y REPORTE
# ============================================================
y_true = []
y_pred = []

for images, labels in test_dataset:
    predicciones = model.predict(images, verbose=0)
    clases_predichas = np.argmax(predicciones, axis=1)

    y_true.extend(labels.numpy())
    y_pred.extend(clases_predichas)

# 🔥 NUEVO
acc = accuracy_score(y_true, y_pred)
print("\nAccuracy score (sklearn):", acc)

print("\nMatriz de confusión:")
print(confusion_matrix(y_true, y_pred))

print("\nReporte de clasificación:")
print(classification_report(y_true, y_pred, target_names=class_names))

# ============================================================
# PASO 12. MODELO FINAL
# ============================================================
print("\nEl mejor modelo quedó guardado en:", ruta_modelo)

Found 2400 files belonging to 12 classes.
Razas de gatos: ['Abyssinian', 'Bengal', 'Birman', 'Bombay', 'British Shorthair', 'Egyptian', 'Maine', 'Persian', 'Ragdoll', 'Russian Blue', 'Siamese', 'Sphynx']
Número de clases: 12

Lotes totales: 75
Lotes train: 52
Lotes validation: 11
Lotes test: 12



Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ sequential (Sequential)              │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lambda (Lambda)                      │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ mobilenetv2_1.00_224 (Functional)    │ (None, 7, 7, 1280)          │       2,257,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 1280)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │         163,968 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 12)                  │           1,548 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,423,500 (9.24 MB)

 Trainable params: 165,516 (646.55 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Epoch 1/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 686ms/step - accuracy: 0.1198 - loss: 2.7880
Epoch 1: val_loss improved from None to 1.95303, saving model to C:\Users\Oscar Ferreira\OneDrive - AUTO LINEAS AMERICA SA DE CV\Escritorio\MCD\5 - APRENDIZAJE PROFUNDO\PROYECTO FINAL\Modelos Mascotas\modelo_razas_gatos_TransferLearning.keras

Epoch 1: finished saving model to C:\Users\Oscar Ferreira\OneDrive - AUTO LINEAS AMERICA SA DE CV\Escritorio\MCD\5 - APRENDIZAJE PROFUNDO\PROYECTO FINAL\Modelos Mascotas\modelo_razas_gatos_TransferLearning.keras
52/52 ━━━━━━━━━━━━━━━━━━━━ 50s 877ms/step - accuracy: 0.1466 - loss: 2.5625 - val_accuracy: 0.4318 - val_loss: 1.9530 - learning_rate: 1.0000e-04
Epoch 2/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 632ms/step - accuracy: 0.2849 - loss: 2.1373
Epoch 2: val_loss improved from 1.95303 to 1.55101, saving model to C:\Users\Oscar Ferreira\OneDrive - AUTO LINEAS AMERICA SA DE CV\Escritorio\MCD\5 - APRENDIZAJE PROFUNDO\PROYECTO FINAL\Modelos Mascotas\modelo_razas_gatos_Tra

## FINE TUNING RAZA GATOS

In [3]:
########################################################################
########## CLASIFICACIÓN DE RAZAS DE GATOS - FINE TUNING ###############
########################################################################

import os
from pathlib import Path

DATA_DIR = Path(os.environ.get("DATA_DIR", "./data"))
MODELS_DIR = Path(os.environ.get("MODELS_DIR", "./models"))
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# ============================================================
# PASO 0. RUTAS
# ============================================================
ruta_gatos = DATA_DIR / "OXFORD" / "GATOS"

# Modelo generado por tu script de Transfer Learning
ruta_modelo_transfer = MODELS_DIR / "modelo_razas_gatos_TransferLearning.keras"

# Nuevo modelo de Fine Tuning
ruta_modelo_finetuning = MODELS_DIR / "modelo_razas_gatos_FineTuning.keras"

IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32
SEED = 42

# ============================================================
# PASO 1. CARGA DEL DATASET
# ============================================================
dataset_completo = tf.keras.utils.image_dataset_from_directory(
    ruta_gatos,
    seed=SEED,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    shuffle=True
)

class_names = dataset_completo.class_names
num_classes = len(class_names)

print("Razas de gatos:", class_names)
print("Número de clases:", num_classes)

# ============================================================
# PASO 2. SEPARAR TRAIN / VALIDATION / TEST
# ============================================================
total_batches = tf.data.experimental.cardinality(dataset_completo).numpy()

train_size = int(0.70 * total_batches)
val_size = int(0.15 * total_batches)
test_size = total_batches - train_size - val_size

train_dataset = dataset_completo.take(train_size)
resto_dataset = dataset_completo.skip(train_size)

val_dataset = resto_dataset.take(val_size)
test_dataset = resto_dataset.skip(val_size)

print("\nLotes totales:", total_batches)
print("Lotes train:", tf.data.experimental.cardinality(train_dataset).numpy())
print("Lotes validation:", tf.data.experimental.cardinality(val_dataset).numpy())
print("Lotes test:", tf.data.experimental.cardinality(test_dataset).numpy())

# ============================================================
# PASO 3. OPTIMIZAR DATASET
# ============================================================
# TensorFlow ajusta automáticamente la mejor cantidad de recursos (CPU / memoria) para cargar datos de forma eficiente.
AUTOTUNE = tf.data.AUTOTUNE

# cache():
# guarda en memoria los datos ya procesados después de la primera pasada,evitando leerlos desde disco en cada época.
# prefetch():prepara el siguiente lote de imágenes mientras la GPU/CPU entrena con el lote actual.
# Resultado: menos tiempos muertos y entrenamiento más rápido.

# En resumen:
# cache = guardar temporalmente datos procesados
# prefetch = cargar datos por adelantado
# AUTOTUNE = TensorFlow decide la mejor configuración

train_dataset = train_dataset.cache().prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.cache().prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.cache().prefetch(buffer_size=AUTOTUNE)

# ============================================================
# PASO 4. CARGAR MODELO DE TRANSFER LEARNING
# ============================================================
print("\nCargando modelo base entrenado con Transfer Learning...")

# load_model(): abre un modelo guardado previamente en disco.
# ruta_modelo_transfer: contiene la ruta del archivo del modelo.
# Este modelo ya fue entrenado con Transfer Learning, es decir, usando conocimiento previo de otra red conocida.
# custom_objects={"preprocess_input": preprocess_input} registra la función de preprocesamiento utilizada por el modelo, necesaria para reconstruirlo correctamente.
# preprocess_input normalmente ajusta pixeles al formato esperado por arquitecturas como MobileNetV2.
# compile=False lo carga sin compilar todavía. Esto permite modificar capas, descongelar bloques o recompilar después para Fine Tuning.

model = load_model(
    ruta_modelo_transfer,
    custom_objects={"preprocess_input": preprocess_input},
    compile=False
)

# ============================================================
# PASO 5. LOCALIZAR Y DESCONGELAR EL MODELO BASE REAL
# ============================================================

# Se crea una variable vacía donde se guardará el modelo base (MobileNetV2) encontrado dentro del modelo completo.
base_model = None

# Recorre todas las capas del modelo cargado. print(...) Muestra nombre y tipo de cada capa. if "mobilenetv2" ...
# Busca la capa principal que corresponde a MobileNetV2, es decir, la red preentrenada usada en Transfer Learning. Cuando la encuentra, la guarda en base_model.

print("\nCapas principales del modelo cargado:")
for layer in model.layers:
    print(layer.name, "->", layer.__class__.__name__)
    if "mobilenetv2" in layer.name.lower():
        base_model = layer

if base_model is None:
    raise ValueError("No se encontró MobileNetV2 dentro del modelo cargado.")

print("\nModelo base encontrado:", base_model.name)

# Descongelar MobileNetV2
# Permite que los pesos internos de MobileNetV2 puedan actualizarse. Antes en Transfer Learning normalmente estaba congelado.
# Ahora podrá reaprender.

base_model.trainable = True

# CONFIGURACIÓN 1 RECOMENDADA
# Indica desde qué capa se empezará a reentrenar.

fine_tune_at = 130

# Congelar las primeras capas y dejar entrenables solo las últimas

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# Las capas finales sí se entrenan nuevamente. Estas capas aprenden rasgos más específicos para tu problema particular.  
    
for layer in base_model.layers[fine_tune_at:]:
    layer.trainable = True

print(f"\nCapas totales en base_model: {len(base_model.layers)}")
print(f"Capas congeladas hasta: {fine_tune_at}")
print("Capas finales desbloqueadas para Fine Tuning.")

# ============================================================
# PASO 6. VERIFICAR CAPAS ENTRENABLES EN BASE MODEL
# ============================================================
entrenables = sum(1 for layer in base_model.layers if layer.trainable)
no_entrenables = sum(1 for layer in base_model.layers if not layer.trainable)

print("\n========================================")
print("RESUMEN DE CAPAS EN BASE_MODEL")
print("========================================")
print("Capas entrenables en base_model    :", entrenables)
print("Capas congeladas en base_model     :", no_entrenables)
print("Capas totales en base_model        :", len(base_model.layers))

print("\nÚltimas capas del base_model:")
ultimas = base_model.layers[-15:]
inicio = len(base_model.layers) - len(ultimas)

for i, layer in enumerate(ultimas):
    idx_real = inicio + i
    print(f"{idx_real:03d} | {layer.name:35s} | trainable={layer.trainable}")

# ============================================================
# PASO 7. RECOMPILAR CON NUEVO LEARNING RATE
# ============================================================
# Después de descongelar capas para Fine Tuning, el modelo debe compilarse nuevamente. Esto actualiza configuración de entrenamiento:
# optimizador, función de pérdida y métricas. optimizer = Adam(learning_rate=3e-5) Adam seguirá ajustando los pesos mediante gradiente descendente.
# learning_rate = 0.00003 Es una tasa muy pequeña porque ahora ya existe un modelo previamente entrenado.Se usa valor bajo para no destruir el conocimiento previo
# de MobileNetV2 y hacer ajustes finos. Si fuera muy grande: podría empeorar el modelo rápidamente.
# loss = "sparse_categorical_crossentropy" Se usa cuando existen múltiples clases y las etiquetas están como números enteros: 0,1,2,3,4...
# Ejemplo:
# 0 = Beagle
# 1 = Husky
# 2 = Pug

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# ============================================================
# PASO 8. CALLBACKS
# ============================================================
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-7,
    verbose=1
)

checkpoint = ModelCheckpoint(
    filepath=ruta_modelo_finetuning,
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=False,
    mode="min",
    verbose=1
)

# ============================================================
# PASO 9. ENTRENAMIENTO FINE TUNING
# ============================================================
history_fine = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=12,
    callbacks=[early_stopping, reduce_lr, checkpoint],
    verbose=1
)

# ============================================================
# PASO 10. CARGAR EL MEJOR MODELO DE FINE TUNING
# ============================================================
print("\nCargando el mejor modelo guardado de Fine Tuning...")
model = load_model(
    ruta_modelo_finetuning,
    custom_objects={"preprocess_input": preprocess_input},
    compile=False
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# ============================================================
# PASO 11. EVALUACIÓN FINAL EN TEST
# ============================================================
test_loss, test_acc = model.evaluate(test_dataset, verbose=0)
print("\nTest loss:", test_loss)
print("Test accuracy:", test_acc)

# ============================================================
# PASO 12. MATRIZ DE CONFUSIÓN Y REPORTE
# ============================================================
y_true = []
y_pred = []

for images, labels in test_dataset:
    predicciones = model.predict(images, verbose=0)
    clases_predichas = np.argmax(predicciones, axis=1)

    y_true.extend(labels.numpy())
    y_pred.extend(clases_predichas)

acc = accuracy_score(y_true, y_pred)
print("\nAccuracy score (sklearn):", acc)

print("\nMatriz de confusión:")
print(confusion_matrix(y_true, y_pred))

print("\nReporte de clasificación:")
print(classification_report(y_true, y_pred, target_names=class_names))

# ============================================================
# PASO 13. MODELO FINAL
# ============================================================
print("\nEl mejor modelo Fine Tuning quedó guardado en:", ruta_modelo_finetuning)

Found 2400 files belonging to 12 classes.
Razas de gatos: ['Abyssinian', 'Bengal', 'Birman', 'Bombay', 'British Shorthair', 'Egyptian', 'Maine', 'Persian', 'Ragdoll', 'Russian Blue', 'Siamese', 'Sphynx']
Número de clases: 12

Lotes totales: 75
Lotes train: 52
Lotes validation: 11
Lotes test: 12

Cargando modelo base entrenado con Transfer Learning...

Capas principales del modelo cargado:
sequential -> Sequential
lambda -> Lambda
mobilenetv2_1.00_224 -> Functional
global_average_pooling2d -> GlobalAveragePooling2D
dropout -> Dropout
dense -> Dense
dropout_1 -> Dropout
dense_1 -> Dense

Modelo base encontrado: mobilenetv2_1.00_224

Capas totales en base_model: 154
Capas congeladas hasta: 130
Capas finales desbloqueadas para Fine Tuning.

RESUMEN DE CAPAS EN BASE_MODEL
Capas entrenables en base_model    : 24
Capas congeladas en base_model     : 130
Capas totales en base_model        : 154

Últimas capas del base_model:
139 | block_15_depthwise_relu             | trainable=True
140 | bloc

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ sequential (Sequential)              │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lambda (Lambda)                      │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ mobilenetv2_1.00_224 (Functional)    │ (None, 7, 7, 1280)          │       2,257,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 1280)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │         163,968 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 12)                  │           1,548 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,423,500 (9.24 MB)

 Trainable params: 1,525,516 (5.82 MB)

 Non-trainable params: 897,984 (3.43 MB)

Epoch 1/12
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 680ms/step - accuracy: 0.7708 - loss: 0.6875
Epoch 1: val_loss improved from None to 0.35080, saving model to C:\Users\Oscar Ferreira\OneDrive - AUTO LINEAS AMERICA SA DE CV\Escritorio\MCD\5 - APRENDIZAJE PROFUNDO\PROYECTO FINAL\Modelos Mascotas\modelo_razas_gatos_FineTuning.keras

Epoch 1: finished saving model to C:\Users\Oscar Ferreira\OneDrive - AUTO LINEAS AMERICA SA DE CV\Escritorio\MCD\5 - APRENDIZAJE PROFUNDO\PROYECTO FINAL\Modelos Mascotas\modelo_razas_gatos_FineTuning.keras
52/52 ━━━━━━━━━━━━━━━━━━━━ 60s 879ms/step - accuracy: 0.7897 - loss: 0.6250 - val_accuracy: 0.8835 - val_loss: 0.3508 - learning_rate: 3.0000e-05
Epoch 2/12
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 667ms/step - accuracy: 0.8300 - loss: 0.4983
Epoch 2: val_loss did not improve from 0.35080
52/52 ━━━━━━━━━━━━━━━━━━━━ 41s 784ms/step - accuracy: 0.8227 - loss: 0.5061 - val_accuracy: 0.8665 - val_loss: 0.3591 - learning_rate: 3.0000e-05
Epoch 3/12
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 667m